In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import os, random
from tqdm import tqdm
from collections import defaultdict

In [2]:
EMBEDDINGS_DIR = "embeddings/"
IDS_DIR        = "ids/"
POI_MAP_PATH   = "photo_poi_mapping.csv"
NUM_CHUNKS     = 768
OUTPUT_PATH    = "quintuplet_embeddings.npy"

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

# Model
INPUT_DIM      = 2048
PROJ_DIM       = 256

# Training
BATCH_SIZE     = 4096    # Large batch — GPU has 24GB free, use it
EPOCHS         = 20
LR             = 1e-3
WEIGHT_DECAY   = 1e-4
EARLY_STOP_PAT = 4
NUM_SAMPLES    = 500_000

# Quintuplet
MARGIN         = None

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Free Memory: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")

Device: cuda
GPU: Tesla V100-SXM2-32GB
Free Memory: 21.02 GB


In [3]:
poi_df = pd.read_csv(POI_MAP_PATH)

# Use correct column name 'photoid' (no underscore)
poi_map = poi_df.set_index('photoid')['poi_id'].to_dict()

all_embeddings, all_poi_ids = [], []

for i in tqdm(range(NUM_CHUNKS), desc="Loading chunks"):
    ep = os.path.join(EMBEDDINGS_DIR, f"embeddings_chunk_{i}.npy")
    ip = os.path.join(IDS_DIR,        f"ids_chunk_{i}.npy")
    if not os.path.exists(ep): continue

    embs = np.load(ep)
    ids  = np.load(ip, allow_pickle=True)  # keep as integers, no .astype(str)
    pois = np.array([poi_map.get(pid, -1) for pid in ids])

    mask = pois != -1
    all_embeddings.append(embs[mask])
    all_poi_ids.append(pois[mask])

all_embeddings = np.concatenate(all_embeddings, axis=0).astype(np.float32)
all_poi_ids    = np.concatenate(all_poi_ids,    axis=0)

print(f"Total valid samples : {all_embeddings.shape[0]}")
print(f"Unique POIs         : {len(np.unique(all_poi_ids))}")

Loading chunks: 100%|█████████████████████████████████████████████████████████████████| 768/768 [01:54<00:00,  6.70it/s]


Total valid samples : 3744663
Unique POIs         : 201623


In [4]:
# Adaptive margin: based on mean pairwise distance of a sample
sample_idx = np.random.choice(len(all_embeddings), min(5000, len(all_embeddings)), replace=False)
sample_emb = torch.tensor(all_embeddings[sample_idx])
sample_emb = F.normalize(sample_emb, p=2, dim=1)

with torch.no_grad():
    dists = torch.cdist(sample_emb, sample_emb, p=2)
    # Use mean distance as margin — data-driven, not arbitrary
    MARGIN = dists.mean().item()

print(f"Adaptive Margin: {MARGIN:.4f}")

# POI → indices lookup (only POIs with ≥3 samples for clean quintuplets)
poi_to_indices = defaultdict(list)
for idx, poi in enumerate(all_poi_ids):
    poi_to_indices[poi].append(idx)

poi_to_indices = {k: v for k, v in poi_to_indices.items() if len(v) >= 3}
valid_pois     = list(poi_to_indices.keys())

print(f"POIs usable for training: {len(valid_pois)}")

Adaptive Margin: 0.9347
POIs usable for training: 201623


In [5]:
def mine_semi_hard_negatives(anchor_emb, neg_pool_embs, pos_dist, margin):
    dists = F.pairwise_distance(
        anchor_emb.unsqueeze(0).expand(len(neg_pool_embs), -1),
        neg_pool_embs
    )
    semi_hard_mask = (dists > pos_dist) & (dists < pos_dist + margin)
    if semi_hard_mask.any():
        return neg_pool_embs[semi_hard_mask][0]
    return neg_pool_embs[dists.argmin()]

In [6]:
class QuintupletDataset(Dataset):
    """
    Vectorized pre-sampling using numpy — eliminates Python loops entirely.
    """
    def __init__(self, embeddings, poi_to_indices, valid_pois, num_samples):
        self.emb = torch.tensor(embeddings, dtype=torch.float32)

        print("Pre-sampling quintuplets (vectorized)...")
        valid_pois_arr = np.array(valid_pois)
        num_pois       = len(valid_pois_arr)

        # Convert poi_to_indices to numpy arrays for fast indexing
        poi_index_arrays = [np.array(poi_to_indices[p]) for p in valid_pois]

        # Sample anchor POI indices all at once
        anchor_poi_idx = np.random.randint(0, num_pois, size=num_samples)

        # Sample 2 different negative POI indices for each sample
        neg_offsets    = np.random.randint(1, num_pois, size=(num_samples, 2))
        neg_poi_idx1   = (anchor_poi_idx + neg_offsets[:, 0]) % num_pois
        neg_poi_idx2   = (anchor_poi_idx + neg_offsets[:, 1]) % num_pois

        print("Sampling within POIs...")
        anchors, pos1s, pos2s, neg1s, neg2s = [], [], [], [], []

        for i in tqdm(range(num_samples), desc="Finalizing"):
            ap   = poi_index_arrays[anchor_poi_idx[i]]
            a, p1, p2 = np.random.choice(ap, 3, replace=len(ap) < 3)
            n1   = np.random.choice(poi_index_arrays[neg_poi_idx1[i]])
            n2   = np.random.choice(poi_index_arrays[neg_poi_idx2[i]])

            anchors.append(a); pos1s.append(p1); pos2s.append(p2)
            neg1s.append(n1);  neg2s.append(n2)

        self.anchors = np.array(anchors)
        self.pos1s   = np.array(pos1s)
        self.pos2s   = np.array(pos2s)
        self.neg1s   = np.array(neg1s)
        self.neg2s   = np.array(neg2s)
        print(f"Done. {num_samples:,} quintuplets ready.")

    def __len__(self):
        return len(self.anchors)

    def __getitem__(self, i):
        return (
            self.emb[self.anchors[i]],
            self.emb[self.pos1s[i]],
            self.emb[self.pos2s[i]],
            self.emb[self.neg1s[i]],
            self.emb[self.neg2s[i]],
        )

NUM_SAMPLES = 500_000
full_dataset = QuintupletDataset(all_embeddings, poi_to_indices, valid_pois, NUM_SAMPLES)

val_size   = int(0.2 * NUM_SAMPLES)
train_size = NUM_SAMPLES - val_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=8, pin_memory=True, persistent_workers=True)

print(f"Train: {train_size:,} | Val: {val_size:,}")
print(f"Steps per epoch: {len(train_loader)}")

Pre-sampling quintuplets (vectorized)...
Sampling within POIs...


Finalizing: 100%|████████████████████████████████████████████████████████████| 500000/500000 [00:47<00:00, 10497.11it/s]


Done. 500,000 quintuplets ready.
Train: 400,000 | Val: 100,000
Steps per epoch: 98


In [7]:
class MEALProjector(nn.Module):
    """
    Residual MLP projector.
    Residual connections prevent vanishing gradients on deep layers
    and stabilize training — critical for high-quality embeddings.
    """
    def __init__(self, input_dim=2048, proj_dim=256):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.GELU(),                  # GELU > ReLU for smoother gradients
            nn.Dropout(0.3),
        )
        self.block2 = nn.Sequential(
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.2),
        )
        self.block3 = nn.Sequential(
            nn.Linear(512, proj_dim),
            nn.BatchNorm1d(proj_dim),
        )

        # Residual shortcut: project input to match block3 output dim
        self.shortcut = nn.Linear(input_dim, proj_dim, bias=False)

    def forward(self, x):
        out = self.block1(x)
        out = self.block2(out)
        out = self.block3(out)
        out = out + self.shortcut(x)   # residual
        return F.normalize(out, p=2, dim=1)

model = MEALProjector(INPUT_DIM, PROJ_DIM).to(DEVICE)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")

MEALProjector(
  (block1): Sequential(
    (0): Linear(in_features=2048, out_features=1024, bias=True)
    (1): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.3, inplace=False)
  )
  (block2): Sequential(
    (0): Linear(in_features=1024, out_features=512, bias=True)
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.2, inplace=False)
  )
  (block3): Sequential(
    (0): Linear(in_features=512, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (shortcut): Linear(in_features=2048, out_features=256, bias=False)
)
Trainable parameters: 3,282,176


In [8]:
class QuintupletLoss(nn.Module):
    """
    4-pair loss from anchor:
      (a,p1,n1), (a,p2,n2), (a,p1,n2), (a,p2,n1)
    Covers both intra-positive and cross-negative relationships.
    """
    def __init__(self, margin):
        super().__init__()
        self.margin = margin

    def forward(self, a, p1, p2, n1, n2):
        d = F.pairwise_distance

        loss = (
            F.relu(self.margin + d(a,p1) - d(a,n1)) +
            F.relu(self.margin + d(a,p2) - d(a,n2)) +
            F.relu(self.margin + d(a,p1) - d(a,n2)) +
            F.relu(self.margin + d(a,p2) - d(a,n1))
        )
        return loss.mean()

criterion = QuintupletLoss(margin=MARGIN)

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# Cosine annealing: smoothly reduces LR → avoids sharp loss spikes
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

In [9]:
# Single GPU — no DataParallel
model = MEALProjector(INPUT_DIM, PROJ_DIM).to(DEVICE)

optimizer  = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
criterion  = QuintupletLoss(margin=MARGIN)

best_val_loss  = float('inf')
patience_count = 0
best_weights   = None

for epoch in range(EPOCHS):
    # --- Train ---
    model.train()
    train_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
        a, p1, p2, n1, n2 = [x.to(DEVICE) for x in batch]
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(a), model(p1), model(p2), model(n1), model(n2))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()

    # --- Validate ---
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]  "):
            a, p1, p2, n1, n2 = [x.to(DEVICE) for x in batch]
            loss = criterion(model(a), model(p1), model(p2), model(n1), model(n2))
            val_loss += loss.item()

    avg_train = train_loss / len(train_loader)
    avg_val   = val_loss   / len(val_loader)
    scheduler.step()

    print(f"Epoch {epoch+1:02d} | Train: {avg_train:.4f} | Val: {avg_val:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

    if avg_val < best_val_loss:
        best_val_loss  = avg_val
        best_weights   = {k: v.clone() for k, v in model.state_dict().items()}
        patience_count = 0
        print("  ✓ Best model saved.")
    else:
        patience_count += 1
        print(f"  No improvement ({patience_count}/{EARLY_STOP_PAT})")
        if patience_count >= EARLY_STOP_PAT:
            print("Early stopping triggered.")
            break

model.load_state_dict(best_weights)
print(f"\nBest Val Loss: {best_val_loss:.4f}")

Epoch 1/20 [Val]  : 100%|███████████████████████████████████████████████████████████████| 25/25 [00:25<00:00,  1.01s/it]


Epoch 01 | Train: 1.7712 | Val: 1.6529 | LR: 0.000994
  ✓ Best model saved.


Epoch 2/20 [Val]  : 100%|███████████████████████████████████████████████████████████████| 25/25 [00:06<00:00,  3.99it/s]


Epoch 02 | Train: 1.6539 | Val: 1.6206 | LR: 0.000976
  ✓ Best model saved.


Epoch 3/20 [Val]  : 100%|███████████████████████████████████████████████████████████████| 25/25 [00:06<00:00,  3.85it/s]


...

Epoch 18 | Train: 1.3120 | Val: 1.5010 | LR: 0.000034
  ✓ Best model saved.


Epoch 19/20 [Val]  : 100%|██████████████████████████████████████████████████████████████| 25/25 [00:06<00:00,  3.87it/s]


Epoch 19 | Train: 1.3065 | Val: 1.5001 | LR: 0.000016
  ✓ Best model saved.


Epoch 20/20 [Val]  : 100%|██████████████████████████████████████████████████████████████| 25/25 [00:06<00:00,  4.12it/s]

Epoch 20 | Train: 1.3014 | Val: 1.5001 | LR: 0.000010
  No improvement (1/4)

Best Val Loss: 1.5001


In [10]:
# Run this in a new cell before restarting
print(f"Free Memory: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")
print(f"Total Memory: {torch.cuda.mem_get_info()[1] / 1e9:.2f} GB")

Free Memory: 24.42 GB
Total Memory: 34.07 GB


In [10]:
import torch

# Save best model weights
MODEL_SAVE_PATH = "meal_projector.pth"
torch.save(model.state_dict(), MODEL_SAVE_PATH)
print(f"Model saved → {MODEL_SAVE_PATH}")

# Verify by reloading
verify_model = MEALProjector(INPUT_DIM, PROJ_DIM).to(DEVICE)
verify_model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=DEVICE))
verify_model.eval()
print("Model reloaded successfully.")

Model saved → meal_projector.pth
Model reloaded successfully.


In [11]:
model.eval()
projected = []
INFER_BATCH = 4096

with torch.no_grad():
    for i in tqdm(range(0, len(all_embeddings), INFER_BATCH), desc="Projecting"):
        batch = torch.tensor(all_embeddings[i:i+INFER_BATCH]).to(DEVICE)
        projected.append(model(batch).cpu().numpy())

projected = np.concatenate(projected, axis=0)

np.save(OUTPUT_PATH, projected)
np.save(OUTPUT_PATH.replace(".npy", "_poi_ids.npy"), all_poi_ids)

print(f"Saved embeddings : {projected.shape}")
print(f"Sample L2 norms  : {np.linalg.norm(projected[:5], axis=1)}")

Projecting: 100%|█████████████████████████████████████████████████████████████████████| 915/915 [00:41<00:00, 21.82it/s]


Saved embeddings : (3744663, 256)
Sample L2 norms  : [1. 1. 1. 1. 1.]
